In [14]:
import pickle
from pathlib import Path
import matplotlib.pyplot as plt
import numpy as np
import xarray as xr
import pandas as pd

In [15]:
# ------------------- Paths -------------------
RUN_DIR = Path("../../2_final_runs/2_us_127_basins/runs")
hydrographs_dir = Path("./hydrographs")
metrics_dir = Path("/home/azureuser/NeuralHydrologyAzure/2_final_runs/2_us_127_basins/ensemble_testing_metrics")

In [16]:
# run_patterns = {
#     "CARAVAN": "precip_prcp_mm_day_seed_*",
#     "CHIRPS":  "precip_prcp_chirps_mm_day_seed_*",
#     "MSWEP":   "precip_prcp_mswep_mm_day_seed_*",
#     "GAUGES":  "precip_prcp_gauge_mm_day_seed_*",
# }

run_patterns = {
    "CAMELS":  "camels_precipitation_seq_270_1_epochs_30_hidden_256_dropout_04_fb_5_seed*",
    "CARAVAN": "total_precipitation_sum_seq_270_1_epochs_30_hidden_256_dropout_04_fb_5_seed*",
    "CHIRPS":  "chirps_precipitation_seq_270_1_epochs_30_hidden_256_dropout_04_fb_5_seed*",
    "MSWEP":   "mswep_precipitation_seq_270_1_epochs_30_hidden_256_dropout_04_fb_5_seed*",
    "CHIRPS_MSWEP":  "chirps_precipitation_mswep_precipitation_seq_270_1_epochs_30_hidden_256_dropout_04_fb_5_seed*"
}

In [17]:
ensemble_by_pattern = {}

for label, pattern in run_patterns.items():
    matched_paths = sorted(RUN_DIR.glob(f"{pattern}/test/model_epoch030/test_results.p"))
    print(f"[{label}] Found {len(matched_paths)} runs: {[p.parts[-4] for p in matched_paths]}")
    
    if not matched_paths:
        print(f"  WARNING: No runs found for '{label}', skipping.")
        continue
    
    all_runs_data = []
    for file_path in matched_paths:
        with open(file_path, "rb") as f:
            all_runs_data.append(pickle.load(f))
    
    ensemble_data = {}
    for basin_id in all_runs_data[0].keys():
        sims = np.stack([
            run[basin_id]['1D']['xr']['streamflow_sim'].values
            for run in all_runs_data
        ], axis=0)
        
        mean_sim = np.mean(sims, axis=0)
        
        xr_ensemble = all_runs_data[0][basin_id]['1D']['xr'].copy(deep=True)
        xr_ensemble['streamflow_sim'].values[:] = mean_sim
        
        ensemble_data[basin_id] = {'1D': {'xr': xr_ensemble}}
    
    ensemble_by_pattern[label] = ensemble_data

[CAMELS] Found 8 runs: ['camels_precipitation_seq_270_1_epochs_30_hidden_256_dropout_04_fb_5_seed111_2904_183346', 'camels_precipitation_seq_270_1_epochs_30_hidden_256_dropout_04_fb_5_seed222_2904_194120', 'camels_precipitation_seq_270_1_epochs_30_hidden_256_dropout_04_fb_5_seed333_2904_204856', 'camels_precipitation_seq_270_1_epochs_30_hidden_256_dropout_04_fb_5_seed444_2904_215643', 'camels_precipitation_seq_270_1_epochs_30_hidden_256_dropout_04_fb_5_seed555_2904_230422', 'camels_precipitation_seq_270_1_epochs_30_hidden_256_dropout_04_fb_5_seed666_3004_001157', 'camels_precipitation_seq_270_1_epochs_30_hidden_256_dropout_04_fb_5_seed777_3004_011932', 'camels_precipitation_seq_270_1_epochs_30_hidden_256_dropout_04_fb_5_seed888_3004_022707']
[CARAVAN] Found 8 runs: ['total_precipitation_sum_seq_270_1_epochs_30_hidden_256_dropout_04_fb_5_seed111_3004_033441', 'total_precipitation_sum_seq_270_1_epochs_30_hidden_256_dropout_04_fb_5_seed222_3004_044222', 'total_precipitation_sum_seq_270_1_

In [18]:
ensemble_by_pattern

{'CAMELS': {'camels_01411300': {'1D': {'xr': <xarray.Dataset> Size: 35kB
    Dimensions:         (date: 2191, time_step: 1)
    Coordinates:
      * date            (date) datetime64[ns] 18kB 2008-10-01 ... 2014-09-30
      * time_step       (time_step) int64 8B 0
    Data variables:
        streamflow_obs  (date, time_step) float32 9kB 0.34 0.31 0.27 ... nan nan nan
        streamflow_sim  (date, time_step) float32 9kB 0.4235 0.4099 ... 1.106 1.001}},
  'camels_01466500': {'1D': {'xr': <xarray.Dataset> Size: 35kB
    Dimensions:         (date: 2191, time_step: 1)
    Coordinates:
      * date            (date) datetime64[ns] 18kB 2008-10-01 ... 2014-09-30
      * time_step       (time_step) int64 8B 0
    Data variables:
        streamflow_obs  (date, time_step) float32 9kB 0.34 0.33 0.33 ... nan nan nan
        streamflow_sim  (date, time_step) float32 9kB 0.5662 0.5353 ... 0.5168}},
  'camels_01487000': {'1D': {'xr': <xarray.Dataset> Size: 35kB
    Dimensions:         (date: 2191, t

In [19]:
# Map labels to their metrics filename (pattern without the _*)
# metrics_filenames = {
#     "CARAVAN": "precip_prcp_mm_day.csv",
#     "CHIRPS":  "precip_prcp_chirps_mm_day.csv",
#     "MSWEP":   "precip_prcp_mswep_mm_day.csv",
#     "GAUGES":  "precip_prcp_gauge_mm_day.csv",
# }

metrics_filenames = {
    "CAMELS":  "camels_precipitation_seq_270_1_epochs_30_hidden_256_dropout_04_fb_5.csv",
    "CARAVAN": "total_precipitation_sum_seq_270_1_epochs_30_hidden_256_dropout_04_fb_5.csv",
    "CHIRPS":  "chirps_precipitation_seq_270_1_epochs_30_hidden_256_dropout_04_fb_5.csv",
    "MSWEP":   "mswep_precipitation_seq_270_1_epochs_30_hidden_256_dropout_04_fb_5.csv",
    "CHIRPS_MSWEP":  "chirps_precipitation_mswep_precipitation_seq_270_1_epochs_30_hidden_256_dropout_04_fb_5.csv"
}

target_basin = "camels_03604000"  # change as needed

for label, basin_dict in ensemble_by_pattern.items():
    out_dir = hydrographs_dir / label
    out_dir.mkdir(parents=True, exist_ok=True)

    metrics_df = pd.read_csv(metrics_dir / metrics_filenames[label], index_col="basin_id")

    for basin_id, data in basin_dict.items():
        if basin_id != target_basin:
            continue

        xr_ds = data['1D']['xr']

        obs  = xr_ds['streamflow_obs'].values.squeeze()
        sim  = xr_ds['streamflow_sim'].values.squeeze()
        time = pd.to_datetime(xr_ds['date'].values)

        if basin_id in metrics_df.index:
            basin_nse = metrics_df.loc[basin_id, "NSE"]
            nse_str = f"{float(basin_nse):.3f}"
        else:
            nse_str = "N/A"

        plt.figure(figsize=(12, 5))
        plt.plot(time, obs, label="Observed", alpha=0.6)
        plt.plot(time, sim, label="Simulated", alpha=0.6)
        plt.xlabel("Date", fontsize=14)
        plt.ylabel("Streamflow (mm/day)", fontsize=14)
        plt.title(f"Test - {basin_id} - {label} - NSE: {nse_str}", fontsize=16)
        plt.legend(fontsize=13)
        plt.grid(True, alpha=0.35)
        plt.tick_params(axis='both', labelsize=12)
        plt.tight_layout()

        plt.savefig(out_dir / f"{basin_id}.png", dpi=150)
        plt.close()

    print(f"[{label}] Saved hydrograph for {target_basin} → {out_dir}")

[CAMELS] Saved hydrograph for camels_03604000 → hydrographs/CAMELS
[CARAVAN] Saved hydrograph for camels_03604000 → hydrographs/CARAVAN
[CHIRPS] Saved hydrograph for camels_03604000 → hydrographs/CHIRPS
[MSWEP] Saved hydrograph for camels_03604000 → hydrographs/MSWEP
[CHIRPS_MSWEP] Saved hydrograph for camels_03604000 → hydrographs/CHIRPS_MSWEP
